<div align="center">

# Day 05: Visualization and Capstone Development

**Digital Twin with Python (VTR UGE 21), hands-on notebook**

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

[Day overview](https://github.com/utkukose/digital-twin-veltech-NB-lecture/tree/main/days/day-05) · [Lecture page](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-05/lecture.html) · [Interactive lab](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-05/lab.html) · [PDF notes](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-05/Day05_Lecture_Notes.pdf)

</div>

## How to use this notebook

This notebook is the hands-on part of the day. The concepts are explained on the [lecture page](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-05/lecture.html), and each section that applies a part of the lecture starts with a link to it. Save your own copy first with File, Save a copy in Drive, then run the cells in order with Shift+Enter. Exercises check your answer as soon as you run them. Each exercise has a reference solution in a collapsed cell: Try first, then open it. Running all cells at once also runs the reference solutions. Nothing needs to be installed.

## 0. Setup

The first cell prepares the environment and defines the helper `check`, which compares an answer with the expected value and prints immediate feedback.

In [ ]:
# Environment: Everything used by this course is preinstalled in Google Colab.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from dataclasses import dataclass, field, asdict
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
plt.rcParams.update({'figure.dpi': 110, 'figure.figsize': (7, 3.6), 'axes.grid': True, 'grid.alpha': 0.3})

def check(name, value, expected, tol=1e-6):
    """Compares an answer with the expected value and prints immediate feedback."""
    if value is None:
        print(f"{name}: not answered yet. Replace None with your answer and run the cell again.")
        return False
    try:
        ok = abs(float(value) - float(expected)) <= tol * max(1.0, abs(float(expected)))
    except (TypeError, ValueError):
        ok = value == expected
    print(f"{name}: " + ("correct." if ok else "not yet. Compare your steps with the lecture and try again."))
    return ok



# The asset: a 48 V lithium-ion battery pack under a duty cycle.
#
#   states   SoC    state of charge, 0 to 1
#            T      cell temperature in degrees Celsius
#            Q      usable capacity in Ah, fades with use
#            R      internal resistance in ohm, grows as capacity fades
#   inputs   I(t)   current demanded by the load in A, positive when discharging
#            u      cooling-fan duty, 0 to 1
#            T_amb  ambient temperature in degrees Celsius
#
# The state of charge changes over minutes, the temperature over tens of minutes and
# the capacity over months.

@dataclass
class PackParams:
    Q0: float = 60.0        # Ah, capacity when new
    R0: float = 0.045       # Ohm, internal resistance when new
    m_cp: float = 9000.0    # J/K, thermal mass of the pack
    hA0: float = 2.2        # W/K, passive convection
    hA1: float = 12.0       # W/K, extra convection at full fan
    k_age: float = 5.5e-8   # Ah lost per A-second at the reference temperature
    Ea_R: float = 3800.0    # K, Arrhenius slope of the ageing law
    T_ref: float = 298.15   # K
    fan_W: float = 25.0     # W, electrical cost of the fan at full duty

    def as_vector(self):
        return np.array([self.R0, self.m_cp, self.hA0, self.hA1])


def pack_derivatives(t, x, I, u, T_amb, p: PackParams):
    """dx/dt for x = [SoC, T, Q, R]. Pure physics, no data, no noise."""
    SoC, T, Q, R = x
    T_K = T + 273.15
    P_gen = I ** 2 * R                                    # ohmic heating, W
    P_cool = (p.hA0 + p.hA1 * u) * (T - T_amb)            # convective loss, W
    arrh = np.exp(-p.Ea_R * (1.0 / T_K - 1.0 / p.T_ref))  # ageing accelerates with heat
    dSoC = -I / (3600.0 * max(Q, 1e-6))
    dT = (P_gen - P_cool) / p.m_cp
    dQ = -p.k_age * abs(I) * arrh
    dR = p.R0 * 0.9 * (p.Q0 - Q) / p.Q0 * 1e-3            # resistance tracks capacity fade
    return np.array([dSoC, dT, dQ, dR])


def duty_cycle(t, kind="urban", seed=RANDOM_STATE):
    """Current demand in amperes as a function of time in seconds."""
    rng = np.random.default_rng(seed)
    if kind == "urban":       # stop-start: bursts of traction, regenerative braking
        base = 14 + 26 * (np.sin(2 * np.pi * t / 420.0) > 0.15)
        regen = -18 * (np.sin(2 * np.pi * t / 420.0 + 0.9) > 0.93)
        return base + regen
    if kind == "highway":     # steady high load
        return 40 + 6 * np.sin(2 * np.pi * t / 1800.0)
    if kind == "idle":
        return 8 + 0 * t
    raise ValueError(kind)



def rk4_step(fun, t, x, dt, *args):
    k1 = fun(t, x, *args)
    k2 = fun(t + dt / 2, x + dt / 2 * k1, *args)
    k3 = fun(t + dt / 2, x + dt / 2 * k2, *args)
    k4 = fun(t + dt, x + dt * k3, *args)
    return x + dt / 6 * (k1 + 2 * k2 + 2 * k3 + k4)


def simulate(p: PackParams, T_end=3600.0, dt=1.0, x0=None, cycle="urban",
             controller=None, T_amb=25.0, seed=RANDOM_STATE):
    """Roll the physical asset forward. `controller(t, x) -> fan duty` closes the loop."""
    x = np.array([0.95, T_amb + 1.0, p.Q0, p.R0]) if x0 is None else np.array(x0, float)
    n = int(T_end / dt)
    ts = np.arange(n + 1) * dt
    X = np.zeros((n + 1, 4)); X[0] = x
    U = np.zeros(n + 1); I_log = np.zeros(n + 1)
    for k in range(n):
        t = ts[k]
        I = float(duty_cycle(t, cycle, seed))
        u = 0.0 if controller is None else float(np.clip(controller(t, x), 0, 1))
        U[k] = u; I_log[k] = I
        x = rk4_step(pack_derivatives, t, x, dt, I, u, T_amb, p)
        x[0] = np.clip(x[0], 0.0, 1.0)
        X[k + 1] = x
    U[-1] = U[-2]; I_log[-1] = I_log[-2]
    return pd.DataFrame({"t": ts, "SoC": X[:, 0], "T": X[:, 1], "Q": X[:, 2],
                         "R": X[:, 3], "u": U, "I": I_log})



@dataclass
class SensorSpec:
    """Turns a true value into a reading: noise, bias, drift, quantisation, dropout and latency."""
    noise_sd: float = 0.0       # gaussian noise, in sensor units
    bias: float = 0.0           # constant offset
    drift_per_hour: float = 0.0 # slow calibration drift
    quantum: float = 0.0        # ADC quantisation step, 0 = none
    dropout: float = 0.0        # probability a sample is lost
    latency_s: float = 0.0      # transport delay


def read_sensor(t, true_values, spec: SensorSpec, rng):
    v = np.asarray(true_values, float).copy()
    v = v + spec.bias + spec.drift_per_hour * (t / 3600.0)
    if spec.noise_sd:
        v = v + rng.normal(0, spec.noise_sd, v.shape)
    if spec.quantum:
        v = np.round(v / spec.quantum) * spec.quantum
    if spec.dropout:
        v = np.where(rng.random(v.shape) < spec.dropout, np.nan, v)
    return v


print('pack model, integrator and sensors ready')

## Python step 5: Classes, records and files

A twin that runs for months must be able to show which settings produced which result. This step freezes a configuration in a dataclass with a fingerprint, writes an event log with a small class and reads it back with pandas &#91;[4](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-05/lecture.html#ref-4), [5](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-05/lecture.html#ref-5)&#93;. A class describes one kind of object: the data that every such object holds, called attributes, and the functions that belong to it, called methods.

### A frozen configuration with a fingerprint

In [ ]:
from dataclasses import dataclass, asdict
import json, hashlib

@dataclass(frozen=True)
class TwinConfig:
    dt_s: float = 10.0
    sync_period_s: float = 60.0
    fan_on_C: float = 42.0
    seed: int = 42

    def fingerprint(self):
        text = json.dumps(asdict(self), sort_keys=True)
        return hashlib.sha256(text.encode()).hexdigest()[:12]

cfg = TwinConfig()
print(cfg)
print("fingerprint:", cfg.fingerprint())

`@dataclass(frozen=True)` creates a class whose attributes cannot be changed after creation. A dataclass lists its attributes with their types and default values, and `cfg = TwinConfig()` creates one object of the class. The method `fingerprint` turns the content into sorted JSON text and hashes it, so the same settings always give the same fingerprint. JSON, the JavaScript Object Notation, is a plain-text format of named values. A hash function turns a text of any length into a fixed number of characters that changes completely when the text changes. Here it is SHA-256, the Secure Hash Algorithm with an output of 256 bits, of which the first 12 characters are kept.

### A frozen object refuses changes

In [ ]:
try:
    cfg.fan_on_C = 40.0
except Exception as err:
    print("refused:", type(err).__name__)
cfg2 = TwinConfig(fan_on_C=40.0)
print("one setting changed, new fingerprint:", cfg2.fingerprint())

`try` runs a block and `except` catches the error it raises, here the refusal of the frozen object. A changed setting requires a new object, and the new object has a different fingerprint, so the change cannot go unnoticed.

### A class that writes a log

In [ ]:
class TwinLog:
    def __init__(self, path, cfg):
        self.path, self.cfg = path, cfg
        open(self.path, "w").close()          # start an empty file

    def write(self, **record):
        record["config"] = self.cfg.fingerprint()
        with open(self.path, "a") as fh:
            fh.write(json.dumps(record) + "\n")

log = TwinLog("twin_log.jsonl", cfg)
for t, T in [(0, 36.0), (10, 36.2), (20, 36.5)]:
    log.write(t=t, T_belief=T, fan=0.0)
print(open("twin_log.jsonl").read())

`**record` collects any keyword arguments into a dictionary, so `write` accepts records with any fields. The mode `"a"` appends to the file, and every record becomes one line of JSON text.

### Reading the log back

In [ ]:
import pandas as pd
back = pd.read_json("twin_log.jsonl", lines=True)
print(back)
print("distinct configurations:", back["config"].nunique())

`pd.read_json(..., lines=True)` reads one record per line into a data frame, the table of pandas with named columns, and `nunique` counts the distinct values of a column. One distinct fingerprint means that the whole log was produced by one configuration.

**Quick check.** Why is the configuration of the twin frozen?

<details><summary>Answer</summary>

So that no setting can change silently while the twin runs: Every change requires a new object with a new fingerprint, which then appears in the log.

</details>

### Exercises for Python step 5

**Exercise 5.1.** Create `TwinConfig(sync_period_s=120.0)` and store its synchronisation period in `answer_51`.

In [ ]:
answer_51 = None   # your answer

check("Exercise 5.1", answer_51, 120.0)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer_51 = TwinConfig(sync_period_s=120.0).sync_period_s

check("Exercise 5.1", answer_51, 120.0)

**Exercise 5.2.** How many records does the data frame `back` contain? Store the number in `n_records`.

In [ ]:
n_records = None   # your answer

check("Exercise 5.2", n_records, 3)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
n_records = len(back)

check("Exercise 5.2", n_records, 3)

**Exercise 5.3.** Do `TwinConfig()` and `cfg` have the same fingerprint? Store the result of the comparison in `same_fp`.

In [ ]:
same_fp = None   # your answer

check("Exercise 5.3", same_fp, True)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
same_fp = TwinConfig().fingerprint() == cfg.fingerprint()

check("Exercise 5.3", same_fp, True)

## 1. The calibrated core

*Lecture: [A dashboard for the operator](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-05/lecture.html#a-dashboard-for-the-operator)*

**From the formulas to the code.** `PackParams` holds the parameters of the energy balance of the lecture. `R0` is the internal resistance $R$ and `m_cp` the heat capacity $m c_p$, and `hA0` and `hA1` are the conductances $hA_0$ and $hA_1$. `TRUE` is the simulated pack with the default values, and `CAL` is the calibrated set of Day 3 that the twin uses in sections 2 to 6. The energy balance itself is the function `pack_derivatives` of the setup cell: `P_gen` is $I^2 R$, `P_cool` is $(hA_0 + hA_1 u)(T - T_{\mathrm{amb}})$, and `dT` is their difference divided by `m_cp`. The line `dSoC` is the rate of the state of charge (SoC). The function `rk4_step`, also in the setup cell, is the fourth-order Runge-Kutta step (RK4) with its four slopes `k1` to `k4`.

In [ ]:
import json, hashlib, time
from pathlib import Path

CAL = PackParams(R0=0.0456, m_cp=9180.0, hA0=2.193, hA1=12.23)   # Day 3: m_cp measured, three parameters fitted
TRUE = PackParams()
print("carried forward: calibrated parameters", CAL.as_vector().round(4))

## 2. A dashboard for the operator

*Lecture: [A dashboard for the operator](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-05/lecture.html#a-dashboard-for-the-operator)*

**From the formulas to the code.** In `run_twin`, `xp` is the state of the pack and `xv` the state of the twin. Each holds the state of charge, the temperature, the capacity and the resistance, so `xv[1]` is the belief $\hat{T}$. The line `resid = m - xv[1]` is the residual $r_k$, and `xv[1] += 0.5 * resid` is the correction with the gain $K = 0.5$. The line `s_cusum = max(0.0, s_cusum + abs(resid) - 0.6)` is the cumulative sum (CUSUM) with the allowance $\kappa = 0.6$, and `s_cusum > 5.0` tests the decision limit $h$. The fan rule is `np.clip((xv[1] - fan_on) / 6, 0, 1)`. In `dashboard`, `rmse` is the root mean square error (RMSE) of the belief against the true temperature over all 540 steps. A reading is taken only when 60 s have passed since the last one, that is every sixth step, and a lost reading (`nan`) leaves the belief and the sum unchanged. The alarm of this section only draws a line in the plot and does not change the fan.

In [ ]:
import ipywidgets as W
from IPython.display import display

def run_twin(T_end=5400.0, dt=10.0, T_amb=35.0, cycle="highway", authority=True,
             sensor_bias=0.0, fan_on=42.0, sync_s=60.0, seed=RANDOM_STATE):
    """One pass of the complete twin. Returns the log the dashboard renders."""
    rng = np.random.default_rng(seed)
    spec = SensorSpec(noise_sd=0.35, bias=sensor_bias, quantum=0.25, dropout=0.02)
    xp = np.array([0.92, T_amb + 1.0, TRUE.Q0, TRUE.R0])
    xv = np.array([0.92, T_amb + 1.0, CAL.Q0, CAL.R0])
    rows = []; last_sync = -1e9; s_cusum = 0.0; alarm_t = None
    for k in range(int(T_end / dt)):
        t = k * dt
        I = float(duty_cycle(t, cycle))
        u = float(np.clip((xv[1] - fan_on) / 6, 0, 1)) if authority else 0.0
        xp = rk4_step(pack_derivatives, t, xp, dt, I, u, T_amb, TRUE)
        xv = rk4_step(pack_derivatives, t, xv, dt, I, u, T_amb, CAL)
        xp[0] = np.clip(xp[0], 0, 1); xv[0] = np.clip(xv[0], 0, 1)
        meas = np.nan
        if t - last_sync >= sync_s:
            last_sync = t
            m = read_sensor(t, [xp[1]], spec, rng)[0]
            if np.isfinite(m):
                meas = m
                resid = m - xv[1]
                s_cusum = max(0.0, s_cusum + abs(resid) - 0.6)
                if alarm_t is None and s_cusum > 5.0:
                    alarm_t = t
                xv[1] += 0.5 * resid                      # partial-gain observer
        rows.append(dict(t=t, I=I, u=u, T_true=xp[1], T_belief=xv[1], T_meas=meas,
                         SoC=xv[0], Q=xv[2], cusum=s_cusum))
    d = pd.DataFrame(rows)
    d.attrs["alarm_t"] = alarm_t
    d.attrs["level"] = "DIGITAL TWIN (two-way)" if authority else "DIGITAL SHADOW (one-way)"
    return d

def dashboard(T_amb=35.0, cycle="highway", authority=True, sensor_bias=0.0, fan_on=42.0):
    d = run_twin(T_amb=T_amb, cycle=cycle, authority=authority,
                 sensor_bias=sensor_bias, fan_on=fan_on)
    err = d.T_belief - d.T_true
    rmse = float(np.sqrt(np.mean(err ** 2)))
    trust = "OK" if rmse < 1.0 else ("DEGRADED" if rmse < 3.0 else "DO NOT TRUST")

    fig = plt.figure(figsize=(11.5, 5.2))
    gs = fig.add_gridspec(2, 3, height_ratios=[2, 1])
    a0 = fig.add_subplot(gs[0, :2])
    a0.plot(d.t / 60, d.T_true, lw=2, color="#111111", label="pack (ground truth)")
    a0.plot(d.t / 60, d.T_belief, lw=1.4, color="#2980b9", label="twin belief")
    a0.scatter(d.t / 60, d.T_meas, s=6, color="#e67e22", label="telemetry")
    a0.axhline(46, ls=":", color="#c0392b")
    if d.attrs["alarm_t"]:
        a0.axvline(d.attrs["alarm_t"] / 60, ls="--", color="#8e44ad", label="residual alarm")
    a0.set_ylabel("T [C]"); a0.legend(fontsize=8, ncol=2); a0.set_xlabel("min")
    a0.set_title(f"{d.attrs['level']}    |    belief RMSE {rmse:.2f} C    |    TRUST: {trust}",
                 fontsize=10)

    a1 = fig.add_subplot(gs[0, 2])
    a1.barh(["SoC %", "capacity %", "fan mean %"],
            [100 * d.SoC.iloc[-1], 100 * d.Q.iloc[-1] / CAL.Q0, 100 * d.u.mean()],
            color=["#2980b9", "#27ae60", "#f39c12"])
    a1.set_xlim(0, 100); a1.set_title("state of the asset", fontsize=9)

    a2 = fig.add_subplot(gs[1, :])
    a2.plot(d.t / 60, d.cusum, color="#c0392b", label="residual CUSUM")
    a2.axhline(5.0, ls="--", lw=.8, color="grey")
    a2.plot(d.t / 60, d.u * 5, color="#f39c12", lw=.9, label="fan duty (scaled)")
    a2.set_xlabel("min"); a2.legend(fontsize=8); a2.set_title("health of the twin", fontsize=9)
    plt.tight_layout(); plt.show()

    print(f"peak pack temperature {d.T_true.max():6.2f} C     seconds above 46 C: "
          f"{int((d.T_true > 46).sum()) * 10}")
    print(f"twin reported above 46 C for {int((d.T_belief > 46).sum()) * 10} s")
    print(f"residual alarm: {'t = %.0f s' % d.attrs['alarm_t'] if d.attrs['alarm_t'] else 'none'}")

try:
    W.interact(dashboard,
               T_amb=W.FloatSlider(35.0, 15.0, 48.0, 1.0),
               cycle=W.Dropdown(options=["highway", "urban", "idle"], value="highway"),
               authority=W.Checkbox(value=True, description="twin may actuate"),
               sensor_bias=W.FloatSlider(0.0, -6.0, 6.0, 0.5),
               fan_on=W.FloatSlider(42.0, 34.0, 50.0, 1.0))
except Exception:
    dashboard()

## 3. Maturity, computed from the state of the twin

*Lecture: [Maturity computed from the state of the twin](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-05/lecture.html#maturity-computed-from-the-state-of-the-twin)*

**From the formulas to the code.** `maturity_level` is the table of the lecture, read from top to bottom. Each `if` is one rule, and the first rule that applies returns its level together with a short note. The argument `rmse_limit=3.0` is the trust limit. With `belief_rmse=None` the second rule is skipped, because an unknown error cannot be compared with the limit.

In [ ]:
def maturity_level(telemetry_ok, actuation_ok, belief_rmse=None, rmse_limit=3.0):
    if not telemetry_ok:
        return "DIGITAL MODEL", "open loop: The predictions are not anchored to the asset and must not be acted on"
    if belief_rmse is not None and belief_rmse > rmse_limit:
        return "DIGITAL MODEL (degraded)", "telemetry present but the model no longer tracks it"
    if not actuation_ok:
        return "DIGITAL SHADOW", "monitoring only: The twin cannot influence the asset"
    return "DIGITAL TWIN", "closed loop: The twin acts on the asset"

cases = [(True, True, 0.4), (True, True, 5.2), (True, False, 0.4), (False, True, 0.4),
         (False, False, None)]
for tel, act, r in cases:
    lvl, note = maturity_level(tel, act, r)
    print(f"telemetry={str(tel):5s} actuation={str(act):5s} rmse="
          f"{('%.1f' % r) if r is not None else ' - ':>4s}  ->  {lvl:26s} {note}")

## 4. A frozen configuration and the event log &#91;[1](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-05/lecture.html#ref-1)&#93;

*Lecture: [Configuration and the event log](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-05/lecture.html#configuration-and-the-event-log)*

**From the formulas to the code.** In `fingerprint`, `asdict(self)` lists the settings, and `json.dumps(..., sort_keys=True)` writes them as text in JSON, the JavaScript Object Notation, with sorted names. `hashlib.sha256(...).hexdigest()` is the hash SHA-256, the Secure Hash Algorithm with 256 bits, and `[:12]` keeps its first 12 characters. `TwinLog.write` puts the fingerprint into the field `fp` of every record, and the records are written to the file in blocks of 200 or when `flush` is called. The log of this section keeps every tenth step (`d.iloc[::10]`), one record per 100 s. The printed sizes count 1024 bytes per kilobyte and are rounded: The 0.1 megabytes (MB) per day stand for 0.07. The logged run comes from `run_twin` of section 2, which uses the parameters `CAL`. The four parameters stored in `TwinConfig` are another set with the same ratios, as the lecture explains under the scorecard, and the twin of section 7 uses them.

In [ ]:
@dataclass(frozen=True)
class TwinConfig:
    """Frozen, so that no code changes it during a run, and fingerprinted, so that a log shows which configuration ran."""
    dt: float = 10.0
    horizon_s: float = 5400.0
    cycle: str = "highway"
    T_amb: float = 35.0
    sync_s: float = 60.0
    fan_on_C: float = 42.0
    observer_gain: float = 0.5
    cusum_k: float = 0.6
    cusum_h: float = 5.0
    R0: float = 0.0426
    m_cp: float = 8571.0
    hA0: float = 2.05
    hA1: float = 11.42
    calibrated_on: str = "2026-08-26 highway 5400 s, 10 s telemetry"
    seed: int = RANDOM_STATE
    def fingerprint(self):
        return hashlib.sha256(json.dumps(asdict(self), sort_keys=True).encode()).hexdigest()[:12]
    def to_json(self, path):
        Path(path).write_text(json.dumps(asdict(self) | {"fingerprint": self.fingerprint()},
                                         indent=2))
    @staticmethod
    def from_json(path):
        d = json.loads(Path(path).read_text()); d.pop("fingerprint", None)
        return TwinConfig(**d)

cfg = TwinConfig()
cfg.to_json("/tmp/twin_config.json")
back = TwinConfig.from_json("/tmp/twin_config.json")
print("fingerprint      :", cfg.fingerprint())
print("round-trips      :", back == cfg)
print("changing one knob:", TwinConfig(fan_on_C=40.0).fingerprint(), "<- different fingerprint")

In [ ]:
class TwinLog:
    """Append-only, flushed in blocks, with the config fingerprint on every record.

    The fingerprint is on every record because logs are concatenated, split and partially
    recovered. A record that does not name the configuration that produced it cannot serve
    as evidence."""
    def __init__(self, cfg, path="/tmp/twin_log.jsonl", block=200):
        self.fp = cfg.fingerprint(); self.path = Path(path); self.block = block
        self.buf = []; self.path.write_text("")
        self.n = 0
    def write(self, **rec):
        self.buf.append(json.dumps({"fp": self.fp, **rec})); self.n += 1
        if len(self.buf) >= self.block:
            self.flush()
    def flush(self):
        if self.buf:
            with self.path.open("a") as f:
                f.write("\n".join(self.buf) + "\n")
            self.buf = []
    def read(self):
        self.flush()
        return pd.DataFrame([json.loads(l) for l in self.path.read_text().splitlines() if l])

log = TwinLog(cfg)
d = run_twin(T_end=cfg.horizon_s, dt=cfg.dt, T_amb=cfg.T_amb, cycle=cfg.cycle,
             fan_on=cfg.fan_on_C, sync_s=cfg.sync_s)
for _, r in d.iloc[::10].iterrows():
    log.write(t=float(r.t), T_true=round(float(r.T_true), 3),
              T_belief=round(float(r.T_belief), 3), u=round(float(r.u), 3))
back = log.read()
print(f"{log.n} records written, {len(back)} read back, "
      f"{back.fp.nunique()} distinct configuration fingerprint")
print(f"log size on disk: {log.path.stat().st_size/1024:.1f} kB for "
      f"{cfg.horizon_s/3600:.1f} h at {cfg.dt*10:.0f} s resolution")
print(f"-> {log.path.stat().st_size/1024 * 24 / (cfg.horizon_s/3600) / 1024:.1f} MB per "
      f"asset per day, {log.path.stat().st_size/1024*24*365/(cfg.horizon_s/3600)/1024/1024:.2f} "
      f"GB per asset per year. Multiply by the size of the fleet before choosing a rate.")

## 5. Verification and validation &#91;[2](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-05/lecture.html#ref-2)&#93;

*Lecture: [Verification and validation](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-05/lecture.html#verification-and-validation)*

**From the formulas to the code.** `exact_T` is the exact solution of the lecture: `T_inf` is $T_\infty$, and `k` is $1 / \tau$, so `np.exp(-k * t)` is $e^{-t / \tau}$. The code verification integrates the temperature equation alone, written as the small function `f`, with `rk4_step`. `errs` holds the errors $e(\Delta t)$ of the five steps, and `order = np.log2(errs[:-1] / errs[1:])` is the observed order of each neighbouring pair. In the solution verification, `ref` is the run with a step of 0.5 s and `abs_err_C` the difference to it. In the validation, `e = model - meas` holds the errors $e_k$, `val_RMSE_C` is their root mean square error (RMSE) and `NOISE` is $\sigma$. The line `implied = np.sqrt(max(0.0, worst ** 2 - NOISE ** 2))` is the implied model error of the worst condition. The readings of the validation carry noise only, without rounding or loss. `validation_experiment` also returns the noise-free temperature `truth`, so `np.sqrt(np.mean((model - truth) ** 2))` gives the true model error, which only a simulation can know.

In [ ]:
# ---- Verification 1: code verification against an exact solution -----------
# With constant current and constant fan, the thermal equation is linear and solvable in closed
# form. An integrator that cannot reproduce this solution cannot be repaired by calibration.
def exact_T(t, T0, I, u, T_amb, p):
    k = (p.hA0 + p.hA1 * u) / p.m_cp
    T_inf = T_amb + I ** 2 * p.R0 / (p.hA0 + p.hA1 * u)
    return T_inf + (T0 - T_inf) * np.exp(-k * t)

p = CAL; T0, I, u, T_amb = 36.0, 40.0, 0.3, 35.0
errs = []
for dt in (400., 200., 100., 50., 25.):
    x = np.array([T0])
    f = lambda t, xx, *a: np.array([(I ** 2 * p.R0 - (p.hA0 + p.hA1 * u) * (xx[0] - T_amb)) / p.m_cp])
    for k in range(int(3600 / dt)):
        x = rk4_step(f, k * dt, x, dt)
    errs.append(abs(float(x[0]) - exact_T(3600, T0, I, u, T_amb, p)))
order = np.log2(np.array(errs[:-1]) / np.array(errs[1:]))
print("Code verification (against the analytic solution)")
print(f"  errors {['%.2e' % e for e in errs]}")
print(f"  observed order of accuracy {order.round(2)}  -> expected 4.0")
print(f"  verdict: {'PASS' if order[-1] > 3.5 else 'FAIL'}\n")

# ---- Verification 2: solution verification (grid convergence) --------------
ref = simulate(CAL, T_end=5400, dt=0.5, cycle="highway", T_amb=35.0)["T"].iloc[-1]
rows = []
for dt in (40., 20., 10., 5., 2.):
    v = simulate(CAL, T_end=5400, dt=dt, cycle="highway", T_amb=35.0)["T"].iloc[-1]
    rows.append((dt, v, abs(v - ref)))
gci = pd.DataFrame(rows, columns=["dt_s", "T_end_C", "abs_err_C"])
print("Solution verification (grid convergence of the deployed configuration)")
print(gci.to_string(index=False, float_format=lambda v: f"{v:.6f}"))
print(f"  discretisation error at the deployed dt = {cfg.dt:.0f} s: "
      f"{float(gci[gci.dt_s == 10.]['abs_err_C'].iloc[0]):.4f} C")

**Exercise A.** What is the last observed order of accuracy in the array `order` of the code verification? Store it in `last_order`.

In [ ]:
last_order = None   # your answer

check("Exercise A", last_order, float(order[-1]))

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
last_order = float(order[-1])

check("Exercise A", last_order, float(order[-1]))

In [ ]:
# ---- Validation: against the real asset, on data not used for calibration --
def validation_experiment(cycle, T_amb, seed):
    rng = np.random.default_rng(seed)
    real = simulate(TRUE, T_end=3600, dt=10.0, cycle=cycle, T_amb=T_amb,
                    controller=lambda t, x: 0.4)
    meas = real["T"].values + rng.normal(0, 0.35, len(real))
    model = simulate(CAL, T_end=3600, dt=10.0, cycle=cycle, T_amb=T_amb,
                     controller=lambda t, x: 0.4)["T"].values
    return meas, model, real["T"].values

rows = []
for cyc, amb, sd in (("highway", 35.0, 1), ("urban", 35.0, 2), ("highway", 22.0, 3),
                     ("urban", 45.0, 4), ("idle", 30.0, 5)):
    meas, model, truth = validation_experiment(cyc, amb, sd)
    e = model - meas
    rows.append(dict(condition=f"{cyc} @ {amb:.0f}C", used_in_calibration=(cyc == "highway" and amb == 35.0),
                     val_RMSE_C=np.sqrt(np.mean(e ** 2)), max_err_C=np.abs(e).max(),
                     mean_err_C=e.mean()))
val = pd.DataFrame(rows)
print("Validation (model vs measured, sensor noise sd = 0.35 C)\n")
print(val.round(3).to_string(index=False))
NOISE = 0.35
worst = float(val.val_RMSE_C.max())
implied = np.sqrt(max(0.0, worst ** 2 - NOISE ** 2))
print(f"\n  worst-case validation RMSE across the envelope : {worst:.3f} C")
print(f"  sensor noise standard deviation                : {NOISE:.3f} C")
print(f"  implied model error, sqrt(RMSE^2 - noise^2)    : {implied:.3f} C")
print(f"  numerical (discretisation) error at dt = 10 s  : "
      f"{float(gci[gci.dt_s == 10.]['abs_err_C'].iloc[0]):.4f} C")

**Exercise B.** The implied model error is `sqrt(RMSE^2 - noise^2)`. Compute it for an RMSE of 0.36 °C and a sensor noise of 0.35 °C, and store it in `e_model`.

In [ ]:
e_model = None   # your answer

check("Exercise B", e_model, float(np.sqrt(0.36 ** 2 - 0.35 ** 2)))

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
e_model = float(np.sqrt(0.36 ** 2 - 0.35 ** 2))

check("Exercise B", e_model, float(np.sqrt(0.36 ** 2 - 0.35 ** 2)))

## 6. The credibility scorecard &#91;[3](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-05/lecture.html#ref-3)&#93;

*Lecture: [The credibility scorecard](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-05/lecture.html#the-credibility-scorecard)*

**From the formulas to the code.** `CREDIBILITY` is a list of rows, each with an element, a status and a line of evidence. The statuses are entered by hand. Three evidence lines take their numbers from section 5 through f-strings. They are the discretisation error at 10 s from the table `gci`, and the largest root mean square error (RMSE) inside and outside the calibration condition from the table `val`. `n_pass` counts the rows with the status `PASS`, and `n_open` counts the open items.

In [ ]:
# ---- Credibility: a scorecard that also lists the open items ----------------
CREDIBILITY = [
 ("Code verification",       "PASS",     "4th-order convergence against an analytic solution"),
 ("Solution verification",   "PASS",     f"discretisation error {float(gci[gci.dt_s==10.]['abs_err_C'].iloc[0]):.4f} C at dt=10 s"),
 ("Validation inside the envelope", "PASS", f"RMSE {val[val.used_in_calibration].val_RMSE_C.max():.2f} C on the calibration condition"),
 ("Validation outside the envelope", "PARTIAL", f"RMSE {val[~val.used_in_calibration].val_RMSE_C.max():.2f} C = sensor noise floor; model error not resolvable"),
 ("Parameter uncertainty",   "PARTIAL",  "four-parameter intervals invalid under the scaling symmetry; m_cp from a separate test (Day 3)"),
 ("Input uncertainty",       "NOT DONE", "ambient temperature and current are treated as known exactly"),
 ("Model form uncertainty",  "NOT DONE", "isothermal pack assumption never tested against a multi-cell model"),
 ("Real hardware",           "NOT DONE", "the entire course runs against a simulated plant"),
 ("Sensor fault coverage",   "PASS",     "bias, drift, dropout, quantisation and latency all exercised"),
 ("Reproducibility",         "PASS",     "frozen config, fingerprint on every log record, fixed seeds"),
]
sc = pd.DataFrame(CREDIBILITY, columns=["element", "status", "evidence"])
print(sc.to_string(index=False))
n_pass = int((sc.status == "PASS").sum())
n_open = len(sc) - n_pass
print(f"\n{n_pass} of {len(sc)} elements PASS; {n_open} are PARTIAL or NOT DONE.")

## 7. The assembled twin on a healthy and a faulted pack

*Lecture: [The assembled twin under a fault](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-05/lecture.html#the-assembled-twin-under-a-fault)*

**From the formulas to the code.** `DigitalTwin` takes its settings from the configuration `cfg`, and its model parameters come from `cfg` as well, not from `CAL` of section 1. `_advance` is one step of the Runge-Kutta method (RK4). `_correct` is the observer, `self.x[1] += self.cfg.observer_gain * r`, followed by the cumulative sum with `cfg.cusum_k` as the allowance $\kappa$. `services` computes the fan rule, the overheat flag `T > 46.0` and the trust flag `self.cusum < self.cfg.cusum_h`. The line `u = svc["fan_duty"] if svc["twin_trustworthy"] else 1.0` is the fail-safe rule. In `report`, `maturity_level(True, self.alarm is None, ...)` treats telemetry as present and actuation as lost once an alarm was raised, and `fan_kWh` is the energy formula with 25 W. The pack, called the plant in the code, is passed to `run` as `plant_params`: `TRUE` for the healthy run, `DEGRADED` for the faulted one, and any other `PackParams` stages another fault. A sensor fault is staged by assigning another `SensorSpec`, for example with a `bias`, to `twin.spec` before the run.

In [ ]:
class DigitalTwin:
    """The components of the five days in one object. Each one was built in an earlier notebook."""

    def __init__(self, cfg: TwinConfig):
        self.cfg = cfg
        self.p = PackParams(R0=cfg.R0, m_cp=cfg.m_cp, hA0=cfg.hA0, hA1=cfg.hA1)
        self.x = None
        self.log = TwinLog(cfg, path="/tmp/capstone_log.jsonl")
        self.cusum = 0.0
        self.alarm = None
        self.spec = SensorSpec(noise_sd=0.35, bias=0.0, quantum=0.25, dropout=0.02)
        self.rng = np.random.default_rng(cfg.seed)

    # ---- Day 2: the simulation core
    def _advance(self, x, t, I, u, params):
        x = rk4_step(pack_derivatives, t, x, self.cfg.dt, I, u, self.cfg.T_amb, params)
        x[0] = np.clip(x[0], 0, 1)
        return x

    # ---- Day 3: the observer
    def _correct(self, meas):
        if np.isfinite(meas):
            r = meas - self.x[1]
            self.x[1] += self.cfg.observer_gain * r
            self.cusum = max(0.0, self.cusum + abs(r) - self.cfg.cusum_k)
            return r
        return np.nan

    # ---- Day 4: the services
    def services(self, t):
        soc, T, Q, R = self.x
        return dict(fan_duty=float(np.clip((T - self.cfg.fan_on_C) / 6, 0, 1)),
                    overheat=bool(T > 46.0),
                    health_pct=float(100 * Q / PackParams().Q0),
                    twin_trustworthy=bool(self.cusum < self.cfg.cusum_h))

    # ---- Day 1: the orchestration loop
    def run(self, plant_params, T_end=None):
        T_end = T_end or self.cfg.horizon_s
        xp = np.array([0.92, self.cfg.T_amb + 1.0, plant_params.Q0, plant_params.R0])
        self.x = np.array([0.92, self.cfg.T_amb + 1.0, self.p.Q0, self.p.R0])
        last, out = -1e9, []
        for k in range(int(T_end / self.cfg.dt)):
            t = k * self.cfg.dt
            I = float(duty_cycle(t, self.cfg.cycle))
            svc = self.services(t)
            u = svc["fan_duty"] if svc["twin_trustworthy"] else 1.0   # fail-safe: full cooling
            xp = self._advance(xp, t, I, u, plant_params)
            self.x = self._advance(self.x, t, I, u, self.p)
            r = np.nan
            if t - last >= self.cfg.sync_s:
                last = t
                r = self._correct(read_sensor(t, [xp[1]], self.spec, self.rng)[0])
                if self.alarm is None and self.cusum > self.cfg.cusum_h:
                    self.alarm = t
            self.log.write(t=t, T_true=round(float(xp[1]), 3),
                           T_belief=round(float(self.x[1]), 3), u=round(u, 3),
                           trust=svc["twin_trustworthy"])
            out.append(dict(t=t, I=I, u=u, T_true=xp[1], T_belief=self.x[1], resid=r,
                            cusum=self.cusum, **svc))
        self.log.flush()
        return pd.DataFrame(out)

    def report(self, d):
        err = d.T_belief - d.T_true
        return {
            "config_fingerprint": self.cfg.fingerprint(),
            "maturity": maturity_level(True, self.alarm is None,
                                       float(np.sqrt(np.mean(err ** 2))))[0],
            "belief_rmse_C": round(float(np.sqrt(np.mean(err ** 2))), 3),
            "peak_pack_T_C": round(float(d.T_true.max()), 2),
            "seconds_over_46C": int((d.T_true > 46).sum() * self.cfg.dt),
            "twin_reported_over_46C_s": int(d.overheat.sum() * self.cfg.dt),
            "fan_kWh": round(float(d.u.sum()) * self.cfg.dt * 25.0 / 3.6e6, 4),
            "trust_lost_at_s": self.alarm,
            "log_records": self.log.n,
        }

twin = DigitalTwin(TwinConfig())
d_ok = twin.run(TRUE)
print("Healthy pack"); print(json.dumps(twin.report(d_ok), indent=2))

In [ ]:
# The same twin, unchanged, runs against a pack that develops a fault.
DEGRADED = PackParams(R0=TRUE.R0 * 1.35, m_cp=TRUE.m_cp, hA0=TRUE.hA0 * 0.45, hA1=TRUE.hA1 * 0.5)
twin2 = DigitalTwin(TwinConfig())
d_bad = twin2.run(DEGRADED)
print("Pack with a cooling fault and raised resistance")
print(json.dumps(twin2.report(d_bad), indent=2))

fig, ax = plt.subplots(1, 2, figsize=(11.5, 3.4), sharey=False)
for a, (d, ttl) in zip(ax, ((d_ok, "healthy pack"), (d_bad, "faulted pack"))):
    a.plot(d.t / 60, d.T_true, lw=2, color="#111111", label="pack")
    a.plot(d.t / 60, d.T_belief, lw=1.3, color="#2980b9", label="twin belief")
    a2 = a.twinx()
    a2.plot(d.t / 60, d.cusum, lw=1, color="#c0392b")
    a2.axhline(TwinConfig().cusum_h, ls="--", lw=.8, color="#c0392b")
    a2.set_ylabel("CUSUM", color="#c0392b")
    a.set_title(ttl, fontsize=9); a.set_xlabel("min"); a.set_ylabel("T [C]")
ax[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

print(f"\ntrust lost at: healthy {twin.alarm}, faulted {twin2.alarm} s")
print(f"reported maturity: healthy {twin.report(d_ok)['maturity']}, "
      f"faulted {twin2.report(d_bad)['maturity']}")

**Exercise C.** How many of the ten scorecard elements pass? Store the number in `n_pass_cards`.

In [ ]:
n_pass_cards = None   # your answer

check("Exercise C", n_pass_cards, int(n_pass))

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
n_pass_cards = int(n_pass)

check("Exercise C", n_pass_cards, int(n_pass))

## 8. Your asset: an operator view and a scorecard

*Lecture: [Your asset and the final project](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-05/lecture.html#your-asset-and-the-final-project)*

Change `ASSET` to `server_rack`, `motor` or `greenhouse`, and set `WORN_FAN = True` to wear out the fan without telling the twin.

**From the formulas to the code.** In `assess_asset`, the inner function `run` is the explicit Euler step of the lecture: `asset["power"]` is the heat input $P(t)$ and `asset["C"]` the heat capacity $C$. `duty` is the fan rule with its threshold 6 degrees below the limit, and `sd` is the noise $\sigma$ of the readings. `T_true` is the real asset and `y` holds its readings. `T` is the twin, a pure model run that no reading corrects, so `rmse`, the root mean square error (RMSE) between `y` and `T`, measures the model error together with the noise. With `worn_fan=True` the real asset runs with `loss=0.5`, half of `hA1`, and the twin keeps `loss=0`. `fine` holds the runs with half and a quarter of the step, sampled at the times of the coarse run, and `order` is the formula of the lecture for the three runs.

In [ ]:
ASSET = "battery"        # choose: battery, server_rack, motor or greenhouse

ASSETS = {
    "battery":     dict(name="battery pack", C=9000.0, hA0=2.2, hA1=12.0, T_amb=35.0, T0=36.0, limit=46.0,
                        horizon=7200.0, dt=5.0, power=lambda t: (40 + 6 * np.sin(2 * np.pi * t / 1800)) ** 2 * 0.045),
    "server_rack": dict(name="server rack", C=1.2e5, hA0=40.0, hA1=400.0, T_amb=24.0, T0=26.0, limit=35.0,
                        horizon=7200.0, dt=10.0, power=lambda t: 3000 + 1500 * np.maximum(0, np.sin(2 * np.pi * t / 3600))),
    "motor":       dict(name="motor winding", C=4000.0, hA0=1.5, hA1=8.0, T_amb=30.0, T0=32.0, limit=90.0,
                        horizon=5400.0, dt=5.0, power=lambda t: (20 + 8 * np.sin(2 * np.pi * t / 1200)) ** 2 * 0.3),
    "greenhouse":  dict(name="greenhouse air", C=3.0e6, hA0=500.0, hA1=4000.0, T_amb=18.0, T0=20.0, limit=32.0,
                        horizon=86400.0, dt=60.0, power=lambda t: 30000 * np.maximum(0, np.sin(2 * np.pi * t / 86400))),
}

def run_asset(a, controller=None, params=None, sensor_sd=0.0, seed=0, T_end=None, dt=None):
    """Explicit Euler for dT/dt = (P(t) - (hA0 + hA1 u)(T - T_amb)) / C; the controller maps a reading to a duty u."""
    p = dict(a); p.update(params or {}); dt = dt or a["dt"]; n = int((T_end or a["horizon"]) / dt)
    t = np.arange(n) * dt; T = np.empty(n); u = np.zeros(n); rng = np.random.default_rng(seed); T[0] = a["T0"]
    for k in range(n - 1):
        reading = T[k] + sensor_sd * rng.normal()
        u[k] = controller(reading) if controller else 0.0
        T[k + 1] = T[k] + dt * (a["power"](t[k]) - (p["hA0"] + p["hA1"] * u[k]) * (T[k] - p["T_amb"])) / p["C"]
    return t, T, u


WORN_FAN = False         # set to True: the real cooling has lost half of its effect, the twin does not know it

def assess_asset(asset, worn_fan=False, seed=RANDOM_STATE):
    """Run the asset and its twin, then verify the integrator, validate the twin against the readings and fill a scorecard."""
    rng = np.random.default_rng(seed); dt = asset["dt"]; n = int(asset["horizon"] / dt); t = np.arange(n) * dt
    sd = 0.01 * (asset["limit"] - asset["T_amb"]); duty = lambda T_: float(np.clip((T_ - (asset["limit"] - 6)) / 6, 0, 1))
    def run(step, loss=0.0):
        m = int(asset["horizon"] / step); T = np.empty(m); u = np.zeros(m); T[0] = asset["T0"]
        for k in range(m - 1):
            u[k] = duty(T[k])
            T[k + 1] = T[k] + step * (asset["power"](k * step) - (asset["hA0"] + asset["hA1"] * (1 - loss) * u[k])
                                      * (T[k] - asset["T_amb"])) / asset["C"]
        return T, u
    T_true, u = run(dt, 0.5 if worn_fan else 0.0); y = T_true + rng.normal(0, sd, n)
    T, _ = run(dt)
    fine = [run(dt / 2 ** j)[0][::2 ** j][:n] for j in (1, 2)]
    order = float(np.log2(np.abs(T - fine[1]).max() / np.abs(fine[0] - fine[1]).max() - 1))
    rmse = float(np.sqrt(np.mean((y - T) ** 2)))
    card = pd.DataFrame([
        ("verification", "observed order of the integrator near 1", f"{order:.2f}", abs(order - 1) < 0.3),
        ("validation", "error against readings within twice the noise", f"{rmse:.3f} vs {sd:.3f}", rmse < 2 * sd),
        ("operation", "peak temperature below the limit", f"{T_true.max():.1f} vs {asset['limit']:.0f}", T_true.max() < asset["limit"]),
        ("operation", "no time above the limit", f"{float((T_true > asset['limit']).sum() * dt):.0f} s", not (T_true > asset["limit"]).any()),
    ], columns=["area", "check", "value", "pass"])
    return t, T, u, y, card

asset = ASSETS[ASSET]
t_d, T_d, u_d, y_d, card = assess_asset(asset, worn_fan=WORN_FAN)
print(f"{asset['name']}{' with a worn fan' if WORN_FAN else ''}: {int(card['pass'].sum())} of {len(card)} checks pass")
print(card.to_string(index=False))
scale, unit = (3600, "h") if asset["horizon"] > 20000 else (60, "min")
fig, ax = plt.subplots(2, 1, figsize=(10, 4.4), sharex=True, gridspec_kw={"height_ratios": [3, 1]})
ax[0].plot(t_d / scale, y_d, ".", ms=2, alpha=.35, label="readings"); ax[0].plot(t_d / scale, T_d, color="#2980b9", label="twin")
ax[0].axhline(asset["limit"], ls="--", color="#c0392b", lw=1, label="limit"); ax[0].legend(fontsize=8); ax[0].set_ylabel("temperature (C)")
ax[1].step(t_d / scale, u_d, where="post", color="#27ae60"); ax[1].set_ylabel("cooling duty"); ax[1].set_xlabel(f"time ({unit})")
ax[0].set_title(f"operator view of the {asset['name']}", fontsize=9); plt.tight_layout(); plt.show()

**Exercise D.** How many checks of the scorecard of your asset pass? Store the number in `n_pass`.

In [ ]:
n_pass = None   # your answer

check("Exercise D", n_pass, int(card["pass"].sum()))

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
n_pass = int(card["pass"].sum())

check("Exercise D", n_pass, int(card["pass"].sum()))

## 9. The final project

*Lecture: [Your asset and the final project](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-05/lecture.html#your-asset-and-the-final-project)*

The brief is in the folder `exams/final` of the repository. It asks for a twin of an asset of your choice with both data flows, a tested core, a calibration with an identifiability check, two elements of the intelligence layer, a dashboard and a filled scorecard.

## Application challenges

Each challenge transfers the ideas of the day to a field of study. Choose the one closest to your programme and solve it in a copy of this notebook.

| Field | Challenge |
|---|---|
| Computer Science and Engineering | Add log rotation to the event log: A new file per day, each beginning with the full configuration, and a reader that concatenates them. |
| Electronics and Communication Engineering | Estimate the bandwidth needed to stream the log of a fleet of 500 packs at one record per second, and propose a compression. |
| Electrical and Electronics Engineering | Extend the scorecard with an element for the electrical model and decide its status from the evidence of the week. |
| Mechanical Engineering | Design the dashboard of a twin of a machine tool: Which three quantities, which alarms, and how is trust shown? |
| Biomedical Engineering | Fill a credibility scorecard for a hypothetical patient twin and identify the element that would block clinical use. |
| Civil Engineering | Compute the implied model error for a bridge strain model whose validation RMSE, the root mean square error, is 12 microstrain with a sensor noise of 10 microstrain, and state what it means. One microstrain is a relative elongation of one millionth. |
| Aeronautical Engineering | Write the fail-safe policy of a twin of an aircraft cooling loop and the condition under which it would be activated. |
| Biotechnology | Design the frozen configuration of a twin of a fermentation process and list the settings whose change must produce a new fingerprint. |

## After the notebook

The self-assessment and the reflection are in the [interactive lab](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-05/lab.html), tabs *Check yourself* and *Reflect and export*. The daily task and the research assignment are on the [day overview](https://github.com/utkukose/digital-twin-veltech-NB-lecture/tree/main/days/day-05).

---

<div align="center">

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

</div>